# DDPM — Paper-to-Code Mock (Colab)

**Paper:** Denoising Diffusion Probabilistic Models (Ho, Jain & Abbeel, 2020) — https://arxiv.org/abs/2006.11239

Hard mock (~30 min). Fill in the `q_sample`, `EpsTheta`, and `sample` stubs, run the 2D generative demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

# Fixed noise schedule (no parameters). alpha_bar is the cumulative product.
T = 400
def make_schedule(T, beta_start=1e-4, beta_end=0.04):
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars
betas, alphas, alpha_bars = make_schedule(T)

## 1. Implement the three core pieces (YOUR TASK)

- `q_sample(x0, t, eps)`: forward jump `x_t = sqrt(abar_t) x0 + sqrt(1-abar_t) eps`.
- `EpsTheta`: a small MLP that takes `(x_t, t)` and predicts the noise (same shape as `x_t`); condition on `t` via the provided sinusoidal embedding.
- `sample(model, n)`: reverse loop from `x_T ~ N(0,I)` down to `x_0` using the DDPM posterior mean, adding `sqrt(beta_t) z` at every step except `t=0`.

In [ ]:
def q_sample(x0, t, eps):
    # TODO: ab = alpha_bars[t].unsqueeze(-1)
    # TODO: return sqrt(ab)*x0 + sqrt(1-ab)*eps
    raise NotImplementedError("fill me in")


def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / half)
    args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


class EpsTheta(nn.Module):
    def __init__(self, data_dim=2, t_dim=32, hidden=128):
        super().__init__()
        self.t_dim = t_dim
        # TODO: build an MLP: Linear(data_dim + t_dim, hidden) -> SiLU -> ... -> Linear(hidden, data_dim)
        raise NotImplementedError("build self.net")

    def forward(self, x_t, t):
        # TODO: temb = timestep_embedding(t, self.t_dim)
        # TODO: return self.net(torch.cat([x_t, temb], dim=-1))
        raise NotImplementedError("fill me in")


@torch.no_grad()
def sample(model, n, data_dim=2):
    x = torch.randn(n, data_dim)  # x_T ~ pure noise
    for i in reversed(range(T)):
        t = torch.full((n,), i, dtype=torch.long)
        eps = model(x, t)
        a, ab = alphas[i], alpha_bars[i]
        # TODO: coef = (1 - a) / sqrt(1 - ab)
        # TODO: mean = (1/sqrt(a)) * (x - coef * eps)
        # TODO: if i > 0: x = mean + sqrt(beta_i) * randn   else: x = mean
        raise NotImplementedError("fill me in")
    return x

## 2. Demonstrate the benefit (2D generative toy task)
Train a denoiser on a **ring** of radius 2, then sample new points from pure noise and check they match the ring.

In [ ]:
R_TRUE, SIGMA = 2.0, 0.05
def sample_ring(n):
    theta = torch.rand(n) * 2 * math.pi
    r = R_TRUE + SIGMA * torch.randn(n)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=-1)

torch.manual_seed(0)
model = EpsTheta()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
losses = []
for step in range(3000):
    x0 = sample_ring(512)
    t = torch.randint(0, T, (512,))
    eps = torch.randn_like(x0)
    loss = F.mse_loss(model(q_sample(x0, t, eps), t), eps)  # L_simple
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

gen = sample(model, 4000)
gen_r = torch.sqrt((gen ** 2).sum(-1))
print(f"loss: {sum(losses[:50])/50:.3f} -> {sum(losses[-50:])/50:.3f}")
print(f"radius mean: gen {gen_r.mean():.3f}  (true {R_TRUE})")
print(f"radius std : gen {gen_r.std():.3f}")
print(f"fraction within 0.3 of ring: {((gen_r - R_TRUE).abs() < 0.3).float().mean():.3f}")

In [ ]:
# OPTIONAL plot (needs matplotlib; the numeric checks above already prove the match).
try:
    import matplotlib.pyplot as plt
    data = sample_ring(2000)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].scatter(data[:, 0].tolist(), data[:, 1].tolist(), s=3); ax[0].set_title("data (ring)")
    ax[1].scatter(gen[:, 0].tolist(), gen[:, 1].tolist(), s=3); ax[1].set_title("generated")
    for a in ax: a.set_aspect("equal"); a.set_xlim(-3, 3); a.set_ylim(-3, 3)
    plt.show()
except ImportError:
    print("matplotlib not available — numeric evidence above is sufficient")

## 3. Sanity checks

In [ ]:
# 1) forward process ends at ~ standard normal
x0 = sample_ring(20000)
tT = torch.full((20000,), T - 1, dtype=torch.long)
xT = q_sample(x0, tT, torch.randn_like(x0))
assert abs(xT.mean().item()) < 0.1 and abs(xT.std().item() - 1.0) < 0.1

# 2) alpha_bar decreasing, in (0,1]
assert (alpha_bars[1:] < alpha_bars[:-1]).all()
assert (alpha_bars > 0).all() and (alpha_bars <= 1.0).all()

# 3) q_sample at t=0 returns ~ x0
x0 = sample_ring(1000); t0 = torch.zeros(1000, dtype=torch.long)
assert (q_sample(x0, t0, torch.randn_like(x0)) - x0).abs().max().item() < 0.05

# 4) eps_theta output shape == input shape
xin = torch.randn(7, 2); tin = torch.randint(0, T, (7,))
assert model(xin, tin).shape == xin.shape

# 5) generated samples match target distribution (right radius, on ring, all sectors)
g = sample(model, 4000); gr = torch.sqrt((g ** 2).sum(-1))
ang = torch.atan2(g[:, 1], g[:, 0])
counts = torch.bincount(((ang + math.pi) / (2 * math.pi) * 8).long().clamp(0, 7), minlength=8)
assert abs(gr.mean().item() - R_TRUE) < 0.15
assert ((gr - R_TRUE).abs() < 0.3).float().mean() > 0.85
assert (counts > 50).all()

# 6) loss decreased + gradient flows
assert sum(losses[-50:]) / 50 < sum(losses[:50]) / 50
m2 = EpsTheta(); x0 = sample_ring(64); t = torch.randint(0, T, (64,)); eps = torch.randn_like(x0)
F.mse_loss(m2(q_sample(x0, t, eps), t), eps).backward()
assert sum(p.grad.abs().sum() for p in m2.parameters() if p.grad is not None).item() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def q_sample(x0, t, eps):
    ab = alpha_bars[t].unsqueeze(-1)
    return torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * eps


class EpsThetaSolution(nn.Module):
    def __init__(self, data_dim=2, t_dim=32, hidden=128):
        super().__init__()
        self.t_dim = t_dim
        self.net = nn.Sequential(
            nn.Linear(data_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x_t, t):
        temb = timestep_embedding(t, self.t_dim)
        return self.net(torch.cat([x_t, temb], dim=-1))


@torch.no_grad()
def sample_solution(model, n, data_dim=2):
    x = torch.randn(n, data_dim)
    for i in reversed(range(T)):
        t = torch.full((n,), i, dtype=torch.long)
        eps = model(x, t)
        a, ab = alphas[i], alpha_bars[i]
        coef = (1.0 - a) / torch.sqrt(1.0 - ab)
        mean = (1.0 / torch.sqrt(a)) * (x - coef * eps)
        x = mean + torch.sqrt(betas[i]) * torch.randn_like(x) if i > 0 else mean
    return x